# Solution 43: Pretrain a decoder-only language model

Start with your `MiniLLM` from assignment 42 at **random weights**. This stage uses only raw text. Predict every next UTF-8 byte with shifted cross-entropy; there are no user/assistant labels or preference rewards. Train the entire model, measure held-out loss, sample text, and save `artifacts/llm/base_reference.pt` for assignment 44.

The offline sentences are a pipeline smoke test. To train on more natural language, manually download the [22.5 MB TinyStories V2 story file](https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt?download=true) and follow [the data guide](../datasets/llm/README.md). The notebook uses converted `pretrain/train.jsonl` if present, otherwise the demo. TinyStories is narrow children's prose; a small byte-level model and a short run cannot guarantee fluent general language.

Implement `LLMTrainer` in the file cell. Basic PyTorch CE, AdamW, clipping, and DataLoader are allowed.

- `loss(logits, labels)`: CE of `logits[:, :-1]` against `labels[:, 1:]`, ignoring `-100`; return differentiable zero if all targets are ignored.
- `train_step(model, optimizer, batches, max_norm)`: token-weighted accumulation over microbatches, one gradient clip and one optimizer update. Reject no valid targets before updating. Return pre-update mean loss as a float.
- `generate(model, input_ids, max_new_tokens, eos_id=2)`: greedy continuation of one nonempty prompt; stop at EOS, budget, or context limit. Preserve prompt and model train/eval mode.

`PretrainDataset` packs each document into context windows and masks padding. Split by document *before* making windows. The judge uses fixed small inputs, checks gradients, updates, generation and repeatability; it does not compare an arbitrary sample to one expected sentence. Run assignment 43 before 44 in the same working directory.


In [ ]:
# Run from the repository root in the same directory as assignment 42's export.
from pathlib import Path
from dataclasses import asdict
import math
import torch
from torch.utils.data import DataLoader
from torch_judge.capstone import LLMConfig, ByteTokenizer, PretrainDataset, read_jsonl, deterministic
from mini_llm_reference import MiniLLM


In [ ]:
%%writefile pretrain_trainer_reference.py
import torch
from torch.nn import functional as F


class LLMTrainer:
    @staticmethod
    def loss(logits, labels):
        targets = labels[:, 1:].reshape(-1)
        if not (targets != -100).any():
            return logits.sum() * 0.0
        return F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), targets, ignore_index=-100)

    @staticmethod
    def train_step(model, optimizer, batches, max_norm=1.0):
        batches = list(batches)
        count = sum(int((labels[:, 1:] != -100).sum()) for _, labels in batches)
        if count == 0:
            raise ValueError('No supervised targets')
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total = 0.0
        for ids, labels in batches:
            n = int((labels[:, 1:] != -100).sum())
            loss = LLMTrainer.loss(model(ids), labels) * (n / count)
            loss.backward()
            total += loss.item()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        optimizer.step()
        return total

    @staticmethod
    @torch.no_grad()
    def generate(model, input_ids, max_new_tokens, eos_id=2):
        if input_ids.ndim != 2 or input_ids.size(0) != 1 or input_ids.size(1) == 0:
            raise ValueError('Generation expects one nonempty prompt')
        if max_new_tokens < 0 or input_ids.size(1) > model.config.max_seq_len:
            raise ValueError('Invalid generation length')
        was_training = model.training
        model.eval()
        try:
            result = input_ids.clone()
            for _ in range(min(max_new_tokens, model.config.max_seq_len - result.size(1))):
                next_id = model(result)[:, -1].argmax(-1, keepdim=True)
                result = torch.cat((result, next_id), dim=1)
                if next_id.item() == eos_id:
                    break
            return result
        finally:
            model.train(was_training)


In [ ]:
from pretrain_trainer_reference import LLMTrainer
from torch_judge import check
check('llm_training')


In [ ]:
DATA = Path('datasets/llm/pretrain')
is_demo = not (DATA / 'train.jsonl').exists()
train_path = DATA / ('demo_train.jsonl' if is_demo else 'train.jsonl')
valid_path = DATA / ('demo_validation.jsonl' if is_demo else 'validation.jsonl')
config = LLMConfig(d_model=64, num_layers=2, hidden_dim=128, max_seq_len=128)
train_data = PretrainDataset(read_jsonl(train_path), config.max_seq_len)
valid_data = PretrainDataset(read_jsonl(valid_path), config.max_seq_len)
train_loader = DataLoader(train_data, batch_size=4, shuffle=False, num_workers=0)
valid_loader = DataLoader(valid_data, batch_size=4, shuffle=False, num_workers=0)
print(f'Data: {train_path}; {len(train_data)} train windows, {len(valid_data)} validation windows')


def evaluate(model, loader):
    was_training = model.training
    model.eval()
    total = count = 0
    try:
        with torch.no_grad():
            for ids, labels in loader:
                n = int((labels[:, 1:] != -100).sum())
                total += LLMTrainer.loss(model(ids), labels).item() * n
                count += n
    finally:
        model.train(was_training)
    return total / count


def run_pretraining(epochs=8, max_steps=None):
    with deterministic(2026):
        model = MiniLLM(config).cpu()
        with torch.no_grad():
            for p in model.parameters():
                if p.ndim >= 2:
                    p.normal_(0, 0.02)
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=.01)
        history, steps = [], 0
        for epoch in range(epochs):
            for ids, labels in train_loader:
                LLMTrainer.train_step(model, optimizer, [(ids, labels)], max_norm=1.0)
                steps += 1
                if max_steps is not None and steps >= max_steps:
                    break
            row = (evaluate(model, train_loader), evaluate(model, valid_loader))
            history.append(row)
            print(f'Epoch {epoch+1}: train={row[0]:.4f}, validation={row[1]:.4f}, perplexity={math.exp(min(20, row[1])):.2f}')
            if max_steps is not None and steps >= max_steps:
                break
        return model, optimizer, history

model, optimizer, history = run_pretraining(epochs=8 if is_demo else 1)


In [ ]:
checkpoint_path = Path('artifacts/llm/base_reference.pt')
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({'config': asdict(config), 'model_state': model.state_dict(),
            'tokenizer': 'utf8_byte_v1', 'stage': 'pretrain'}, checkpoint_path)
print('Saved:', checkpoint_path)


In [ ]:
tokenizer = ByteTokenizer()
seed_text = 'Once upon a time'
seed_ids = torch.tensor([[tokenizer.bos_id] + tokenizer.encode(seed_text)])
completion = LLMTrainer.generate(model, seed_ids, max_new_tokens=40)
print(tokenizer.decode(completion[0].tolist()))
